Authentication in Azure Machine Learning
This notebook shows you how to authenticate to your Azure ML Workspace using

1. Interactive Login Authentication
2. Azure CLI Authentication
3. Managed Service Identity (MSI) Authentication
4. Service Principal Authentication
5. Token Authentication

The interactive authentication is suitable for local experimentation on your own computer. Azure CLI authentication is suitable if you are already using Azure CLI for managing Azure resources, and want to sign in only once. The MSI and Service Principal authentication are suitable for automated workflows, for example as part of Azure Devops build.

In [1]:
from azureml.core import Workspace

Interactive Authentication
Interactive authentication is the default mode when using Azure ML SDK.

When you connect to your workspace using workspace.from_config, you will get an interactive login dialog.

In [3]:
ws = Workspace.from_config()

Performing interactive authentication. Please follow the instructions on the terminal.


The default web browser has been opened at https://login.microsoftonline.com/organizations/oauth2/v2.0/authorize. Please continue the login in the web browser. If no web browser is available or if the web browser fails to open, use device code flow with `az login --use-device-code`.


Interactive authentication successfully completed.


Also, if you explicitly specify the subscription ID, resource group and workspace name, you will get the dialog.

In [ ]:
ws = Workspace(subscription_id="my-subscription-id",
               resource_group="my-ml-rg",
               workspace_name="my-ml-workspace")

Note the user you're authenticated as must have access to the subscription and resource group. If you receive an error

AuthenticationException: You don't have access to xxxxxx-xxxx-xxx-xxx-xxxxxxxxxx subscription. All the subscriptions that you have access to = ...
check that the you used correct login and entered the correct subscription ID.

In some cases, you may see a version of the error message containing text: All the subscriptions that you have access to = []

In such a case, you may have to specify the tenant ID of the Azure Active Directory you're using. An example would be accessing a subscription as a guest to a tenant that is not your default. You specify the tenant by explicitly instantiating InteractiveLoginAuthentication with Tenant ID as argument. The Tenant ID can be found, for example, from https://portal.azure.com under Azure Active Directory, Properties as Directory ID.

In [ ]:
from azureml.core.authentication import InteractiveLoginAuthentication

interactive_auth = InteractiveLoginAuthentication(tenant_id="my-tenant-id")

ws = Workspace(subscription_id="my-subscription-id",
               resource_group="my-ml-rg",
               workspace_name="my-ml-workspace",
               auth=interactive_auth)

Despite having access to the workspace, you may sometimes see the following error when retrieving it:

You are currently logged-in to xxxxxxxx-xxx-xxxx-xxxx-xxxxxxxxxxxx tenant. You don't have access to xxxxxx-xxxx-xxx-xxx-xxxxxxxxxx subscription, please check if it is in this tenant.
This error sometimes occurs when you are trying to access a subscription to which you were recently added. In this case, you need to force authentication again to avoid using a cached authentication token that has not picked up the new permissions. You can do so by setting force=true on the InteractiveLoginAuthentication() object's constructor as follows:

In [ ]:
forced_interactive_auth = InteractiveLoginAuthentication(tenant_id="my-tenant-id", force=True)

ws = Workspace(subscription_id="my-subscription-id",
               resource_group="my-ml-rg",
               workspace_name="my-ml-workspace",
               auth=forced_interactive_auth)

Azure CLI Authentication
If you have installed azure-cli package, and used az login command to log in to your Azure Subscription, you can use AzureCliAuthentication class.

Note that interactive authentication described above won't use existing Azure CLI auth tokens.

In [6]:
from azureml.core.authentication import AzureCliAuthentication

cli_auth = AzureCliAuthentication()

ws = Workspace(subscription_id="3a1bee8e-cbe2-4d43-a1af-7b76f7f45414",
               resource_group="data-engineering-ml",
               workspace_name="my-ml-workspacedata-engineering-ml-001",
               auth=cli_auth)

print("Found workspace {} at location {}".format(ws.name, ws.location))

AuthenticationException: AuthenticationException:
	Message: Could not retrieve user token. Please run 'az login'
	InnerException User 'andria2191@outlook.com' does not exist in MSAL token cache. Run `az login`.
	ErrorResponse 
{
    "error": {
        "code": "UserError",
        "inner_error": {
            "code": "Authentication"
        },
        "message": "Could not retrieve user token. Please run 'az login'"
    }
}